# CNNs on iCoSimal V3 Dataset

This notebook documents the full experimental pipeline for the CNN experiments on the iCoSimal V3 dataset.

The goal is to:
- build a clean and reproducible data pipeline,
- start with simple baseline models,
- and progressively explore model complexity, hyperparameters, and regularization techniques.

The dataset contains 30,000 images from 10 balanced animal classes, split into:
- 24,000 training images
- 6,000 validation images

To enable faster experimentation, we initially resize images to 128 × 128 pixels.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from p01_cnn_icosimal import datasets, experiment_runner, models

In [ ]:
# Project paths
REPO_ROOT = Path.cwd().parent
DATA_ROOT = REPO_ROOT / "data" / "icosimal_img_class_03" / "data_uniform_224_224_sets"
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "validate"

# Global configuration
SEED = 42
PIN_MEMORY = torch.cuda.is_available()

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE       :", DEVICE)
print("DATA_ROOT    :", DATA_ROOT)
print("TRAIN_DIR    :", TRAIN_DIR, "(", TRAIN_DIR.exists(), ")")
print("VAL_DIR      :", VAL_DIR, "(", VAL_DIR.exists(), ")")
print("SEED         :", SEED)

---
## Data pipeline overview

Before training any model, we set up a proper data pipeline consisting of:

1. Computing normalization statistics (mean and standard deviation) on the training set
2. Building transformations (resize, normalization, optional augmentation)
3. Creating datasets and dataloaders
4. Inspecting batches to verify correctness

We first demonstrate the dataloader with augmentation enabled to understand its behavior.
Afterwards, we define a clean baseline dataloader **without augmentation**, which will be used for initial training experiments.

### Compute normalization statistics

We compute the channel-wise mean and standard deviation using the training split only.

This ensures that no information from the validation set leaks into the preprocessing pipeline.

In [ ]:
# Compute normalization statistics once from the training split
mean, std = datasets.compute_train_mean_std(
    train_dir=TRAIN_DIR,
    image_size=128,
    batch_size=64,
    num_workers=8,
    pin_memory=PIN_MEMORY,
)

print("Mean:", mean)
print("Std :", std)

### Demonstration of the dataloader with augmentation

To better understand the data pipeline, we first build a dataloader with basic augmentation enabled.

This allows us to:
- verify that transformations work correctly,
- visually inspect the effect of augmentation,
- confirm that labels and images are aligned.

This dataloader is **only used for inspection**, not for baseline training.

In [ ]:
train_loader_demo, val_loader_demo, class_names_demo, _, _ = datasets.build_dataloaders(
    data_root=DATA_ROOT,
    mean=mean,
    std=std,
    image_size=128,
    batch_size=64,
    num_workers=8,
    pin_memory=PIN_MEMORY,
    use_flip=True,
    randaugment_num_ops=2,
    randaugment_magnitude=9,
    seed=SEED,
)

In [ ]:
demo_images, demo_labels = next(iter(train_loader_demo))

print("Demo batch images shape:", demo_images.shape)
print("Demo batch labels shape:", demo_labels.shape)
print("Demo classes           :", class_names_demo)

datasets.show_batch(
    images=demo_images,
    labels=demo_labels,
    class_names=class_names_demo,
    mean=mean,
    std=std,
    max_images=8,
    figsize=(16, 4),
)

### Baseline dataloader (no augmentation)

For the initial experiments, we use a clean baseline dataloader **without any data augmentation**.

This is important because:
- it provides a clear reference point,
- it allows us to study overfitting and underfitting behavior,
- and it ensures that improvements can later be attributed to specific techniques (e.g., augmentation).

All initial models will be trained using this setup.

In [ ]:
data_bundle = datasets.build_data_bundle(
    data_root=DATA_ROOT,
    image_size=128,
    batch_size=64,
    num_workers=8,
    pin_memory=PIN_MEMORY,
    use_flip=False,
    randaugment_num_ops=0,
    randaugment_magnitude=9,
    mean=mean,
    std=std,
    dataset_name="iCoSimal_V3_128_baseline",
    seed=SEED,
)

class_names = data_bundle.class_names

print("Number of classes     :", len(class_names))
print("Class names           :", class_names)
print("Training samples      :", len(data_bundle.train_dataset))
print("Validation samples    :", len(data_bundle.val_dataset))
print("Bundle data config keys:")
print(sorted(data_bundle.data_config.keys()))

### Final sanity check

We inspect one batch from the baseline dataloader to confirm:
- correct tensor shapes,
- correct label mapping,
- correct normalization.

In [ ]:
images, labels = next(iter(data_bundle.train_loader))

print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)
print("Batch dtype       :", images.dtype)
print("Batch device      :", images.device)
print("First labels      :", labels.tolist()[:8])
print("First classes     :", [class_names[label] for label in labels.tolist()[:8]])

datasets.show_batch(
    images=images,
    labels=labels,
    class_names=class_names,
    mean=data_bundle.mean,
    std=data_bundle.std,
    max_images=8,
    figsize=(16, 4),
)

---
## Baseline model

We start with a small baseline CNN in order to establish a clear reference point.
The purpose of this first model is not to maximize performance, but to create a simple and reproducible baseline for later comparisons.

This baseline will be used to:
- observe initial learning behavior,
- identify underfitting or overfitting,
- and motivate later increases in model depth and complexity.

In [ ]:
model = models.test_cnn.TestCNN(num_classes=len(class_names))

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [ ]:
result = experiment_runner.run_experiment(
    data_bundle=data_bundle,
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    num_epochs=15,
    scheduler=scheduler,
    run_name_suffix="baseline_no_aug_128",
    tags=["128x128", "no_aug"],
    notes="Clean baseline run on iCoSimal V3 at 128x128 without augmentation.",  # can be removed
    group="baseline",  # can be removed
    job_type="train",  # can be removed
    seed=SEED,
    deterministic=True,
)

# TODO: Evaluation

In [ ]:
print("Best epoch        :", result["best_epoch"])
print("Best val accuracy :", f"{result['best_val_accuracy']:.4f}")
print("Best val loss     :", f"{result['best_val_loss']:.4f}")

In [ ]:
history = result["history"]
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], label="train_loss")
plt.plot(epochs, history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curves")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_accuracy"], label="train_accuracy")
plt.plot(epochs, history["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy curves")
plt.legend()
plt.grid(True)
plt.show()